In [3]:
import cv2
import mediapipe as mp
import numpy as np
from deepface import DeepFace
import socket
import pickle
import time
#logic to create socket connection with unity
soc = socket.socket()
hostname = "localhost"
port = 5000
soc.bind((hostname, port))
soc.listen(5)
conn, addr = soc.accept()
print("Device Connected")
old_msg=''
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
holistic = mp_holistic.Holistic(
    static_image_mode=False,
    min_detection_confidence=0.65,#used to be 0.5, increased for better accuracy
    model_complexity=1
)

face_ids = {}
frame_count = 0
cap = cv2.VideoCapture(0)

while cap.isOpened():
    _, frame = cap.read()
    msg=''
    try:
        
     
        f_frame = cv2.resize(frame, (480, 640))
        frame_rgb = cv2.cvtColor(f_frame, cv2.COLOR_BGR2RGB)
        frame_count += 1
        # if frame_count % 60 == 0:
        #     face_encodings = DeepFace.represent(
        #         f_frame,
        #         model_name="Facenet",
        #         enforce_detection=False
        #     )
        #     if len(face_encodings) > 0:
        #         face_id = tuple(face_encodings[0]["embedding"])
        #         match = None
        #         for k in face_ids:
        #             if np.linalg.norm(np.array(face_id) - np.array(k)) < 15:
        #                 match = face_ids[k]
        #                 msg = "Known face recognized: " + match
        #                 break
        #         if match is None:
        #             face_ids[face_id] = "Person " + str(len(face_ids) + 1)
        #             msg = "New face detected: " + face_ids[face_id]
        #             print("New face detected: " + face_ids[face_id])
        #         else:
        #             print("Known face recognized: " + match)
        results = holistic.process(frame_rgb)
        annotated_image = f_frame.copy()
        image_height, image_width, _ = frame_rgb.shape
        # mp_pose = mp.solutions.pose
        # x=int(results.pose.landmark[mp_pose.PoseLandmark.RIGHT_WRIST].x * image_width)
        # y=int(results.pose.landmark[mp_pose.PoseLandmark.RIGHT_WRIST].y * image_height)
        for face_id_key, name in face_ids.items():
            cv2.putText(annotated_image, name, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        mp_drawing.draw_landmarks(annotated_image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
        mp_drawing.draw_landmarks(annotated_image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
        mp_drawing.draw_landmarks(
            annotated_image,
            results.face_landmarks,
            mp_holistic.FACEMESH_TESSELATION,
            landmark_drawing_spec=None,
            connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_tesselation_style()
        )
        mp_drawing.draw_landmarks(
            annotated_image,
            results.pose_landmarks,
            mp_holistic.POSE_CONNECTIONS,
            landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style()
        )
        cv2.imshow("Output", annotated_image)
        #logic to send msg to unity
        if msg!='' and msg != old_msg:  # only send when there's actually something
            conn.send(msg.encode('utf-8'))
        old_msg = msg

        if msg == pickle.dumps("exit"):
            break
    except Exception as e:
        print(e)
    if cv2.waitKey(1) == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

Device Connected


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import socket
import pickle
import socket
import asyncio
from bleak import BleakScanner

SERVER_HOST        = "0.0.0.0"
SERVER_PORT        = 5000
TUIO_NOTE          = "Remember to start your TUIO simulator/tracker on port 3333"
PHONE_CAMERA_URL   = ""
PHONE_BT_NAME      = "Phone"

soc = socket.socket()
hostname = "localhost"
port = 5000
soc.bind((hostname, port))
soc.listen(5)
Allpoints=[]
mp_pose = mp.solutions.pose
conn, addr = soc.accept()
print("Device Connected")
old_msg=''
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
holistic = mp_holistic.Holistic(
    static_image_mode=False,
    min_detection_confidence=0.65,
    model_complexity=1
)

face_ids = {}
frame_count = 0
from dollarpy import Recognizer, Template, Point
SwipeLeft = Template('SwipeLeft', [
Point(111,481, 1),
Point(393,470, 1),
Point(110,480, 1),
Point(390,471, 1),
Point(110,480, 1),
Point(389,471, 1),
Point(109,479, 1),
Point(387,471, 1),
Point(109,479, 1),
Point(387,470, 1),
Point(109,479, 1),
Point(386,470, 1),
Point(109,479, 1),
Point(386,470, 1),
Point(109,479, 1),
Point(387,470, 1),
Point(109,479, 1),
Point(387,470, 1),
Point(109,479, 1),
Point(387,470, 1),
Point(109,479, 1),
Point(388,470, 1),
Point(109,479, 1),
Point(388,469, 1),
Point(109,479, 1),
Point(389,469, 1),
Point(109,480, 1),
Point(389,469, 1),
Point(108,480, 1),
Point(390,469, 1),
Point(108,480, 1),
Point(390,469, 1),
Point(108,481, 1),
Point(390,469, 1),
Point(108,481, 1),
Point(390,469, 1),
Point(107,482, 1),
Point(390,469, 1),
Point(107,483, 1),
Point(390,469, 1),
Point(107,483, 1),
Point(390,469, 1),
Point(107,484, 1),
Point(391,470, 1),
Point(107,484, 1),
Point(391,470, 1),
Point(107,484, 1),
Point(391,470, 1),
Point(107,484, 1),
Point(391,471, 1),
Point(106,484, 1),
Point(391,471, 1),
Point(106,485, 1),
Point(392,472, 1),
Point(106,485, 1),
Point(392,472, 1),
Point(106,485, 1),
Point(392,473, 1),
Point(105,486, 1),
Point(392,473, 1),
Point(105,486, 1),
Point(392,474, 1),
Point(104,486, 1),
Point(392,474, 1),
Point(104,486, 1),
Point(392,474, 1),
Point(104,486, 1),
Point(392,474, 1),
Point(104,486, 1),
Point(392,474, 1),
Point(104,486, 1),
Point(393,474, 1),
Point(104,487, 1),
Point(393,474, 1),
Point(104,487, 1),
Point(393,474, 1),
Point(104,487, 1),
Point(393,474, 1),
Point(104,487, 1),
Point(393,474, 1),
Point(104,488, 1),
Point(393,474, 1),
Point(104,488, 1),
Point(393,474, 1),
Point(104,488, 1),
Point(393,474, 1),
Point(104,488, 1),
Point(393,474, 1),
Point(104,488, 1),
Point(393,474, 1),
Point(104,489, 1),
Point(393,474, 1),
Point(104,489, 1),
Point(393,474, 1),
Point(104,489, 1),
Point(393,474, 1),
Point(104,490, 1),
Point(393,474, 1),
Point(104,490, 1),
Point(394,474, 1),
Point(104,490, 1),
Point(394,474, 1),
Point(104,490, 1),
Point(394,474, 1),
Point(104,491, 1),
Point(394,474, 1),
Point(105,491, 1),
Point(394,474, 1),
Point(105,491, 1),
Point(394,475, 1),
Point(106,492, 1),
Point(394,476, 1),
Point(106,492, 1),
Point(394,476, 1),
Point(197,358, 1),
Point(329,367, 1),
Point(206,349, 1),
Point(319,356, 1),
Point(206,348, 1),
Point(315,354, 1),
Point(183,383, 1),
Point(327,374, 1),
Point(178,382, 1),
Point(337,373, 1),
Point(176,381, 1),
Point(343,364, 1),
Point(203,320, 1),
Point(339,374, 1),
Point(206,317, 1),
Point(338,377, 1),
Point(207,317, 1),
Point(335,376, 1),
Point(201,334, 1),
Point(319,303, 1),
Point(197,351, 1),
Point(326,309, 1),
Point(194,443, 1),
Point(323,274, 1),
Point(193,453, 1),
Point(321,266, 1),
Point(182,463, 1),
Point(319,262, 1),
Point(122,492, 1),
Point(321,255, 1),
Point(116,496, 1),
Point(322,263, 1),
Point(113,498, 1),
Point(323,271, 1),
Point(106,504, 1),
Point(323,268, 1),
Point(104,507, 1),
Point(324,267, 1),
Point(106,512, 1),
Point(324,268, 1),
Point(108,516, 1),
Point(325,271, 1),
Point(106,517, 1),
Point(326,274, 1),
Point(106,521, 1),
Point(328,275, 1),
Point(105,528, 1),
Point(330,276, 1),
Point(104,528, 1),
Point(330,274, 1),
Point(104,532, 1),
Point(329,274, 1),
Point(104,536, 1),
Point(330,274, 1),
Point(103,534, 1),
Point(330,275, 1),
Point(99,534, 1),
Point(331,275, 1),
Point(98,533, 1),
Point(331,275, 1),
Point(98,532, 1),
Point(332,276, 1),
Point(98,532, 1),
Point(333,276, 1),
Point(98,533, 1),
Point(333,276, 1),
Point(98,533, 1),
Point(334,276, 1),
Point(99,535, 1),
Point(334,276, 1),
Point(99,535, 1),
Point(335,276, 1),
Point(99,536, 1),
Point(335,277, 1),
Point(99,537, 1),
Point(335,278, 1),
Point(99,539, 1),
Point(336,278, 1),
Point(99,541, 1),
Point(337,278, 1),
Point(99,542, 1),
Point(338,278, 1),
Point(101,545, 1),
Point(342,278, 1),
Point(101,547, 1),
Point(345,278, 1),
Point(102,549, 1),
Point(346,278, 1),
Point(103,554, 1),
Point(352,278, 1),
Point(103,553, 1),
Point(356,278, 1),
Point(101,555, 1),
Point(355,278, 1),
Point(99,555, 1),
Point(358,279, 1),
Point(99,555, 1),
Point(360,280, 1),
Point(99,555, 1),
Point(362,280, 1),
Point(99,555, 1),
Point(363,281, 1),
Point(99,551, 1),
Point(365,282, 1),
Point(99,547, 1),
Point(365,282, 1),
Point(99,546, 1),
Point(365,282, 1),
Point(99,534, 1),
Point(364,264, 1),
Point(99,533, 1),
Point(365,241, 1),
Point(100,532, 1),
Point(365,243, 1),
Point(98,529, 1),
Point(372,253, 1),
Point(97,527, 1),
Point(372,258, 1),
Point(96,522, 1),
Point(377,258, 1),
Point(94,516, 1),
Point(382,260, 1),
Point(92,513, 1),
Point(386,261, 1),
Point(92,509, 1),
Point(414,479, 1),
Point(93,507, 1),
Point(420,490, 1),
Point(93,506, 1),
Point(417,493, 1),
Point(94,505, 1),
Point(420,456, 1),
Point(94,505, 1),
Point(419,426, 1),
Point(95,504, 1),
Point(419,422, 1),
Point(95,501, 1),
Point(419,489, 1),
Point(95,499, 1),
Point(419,494, 1),
Point(95,497, 1),
Point(420,493, 1),
Point(96,495, 1),
Point(419,481, 1),
Point(97,494, 1),
Point(418,478, 1),
Point(97,494, 1),
Point(418,477, 1),
Point(97,494, 1),
Point(417,476, 1),
Point(97,494, 1),
Point(417,475, 1),
Point(97,494, 1),
Point(416,474, 1),
Point(97,494, 1),
Point(416,475, 1),
Point(97,494, 1),
Point(416,476, 1),
Point(97,494, 1),
Point(415,477, 1),
Point(97,494, 1),
Point(415,478, 1),
Point(97,495, 1),
Point(415,478, 1),
Point(97,495, 1),
Point(415,477, 1),
Point(97,494, 1),
Point(415,477, 1),
Point(97,494, 1),
Point(415,477, 1),
Point(97,494, 1),
Point(414,477, 1),
Point(97,494, 1),
Point(414,476, 1),
Point(97,494, 1),
Point(414,476, 1),
Point(96,494, 1),
Point(414,476, 1),
Point(96,494, 1),
Point(414,476, 1),
Point(96,494, 1),
Point(414,476, 1),
Point(96,494, 1),
Point(413,477, 1),
Point(97,495, 1),
Point(412,478, 1),
Point(97,495, 1),
Point(411,480, 1),
Point(97,495, 1),
Point(410,480, 1),
Point(97,495, 1),
Point(408,481, 1),
Point(97,495, 1),
Point(408,482, 1),
Point(97,495, 1),
Point(407,482, 1),
Point(97,495, 1),
Point(406,483, 1),
Point(97,494, 1),
Point(404,483, 1),
Point(97,494, 1),
Point(404,483, 1),
Point(97,493, 1),
Point(402,483, 1),
Point(97,493, 1),
Point(401,483, 1),
Point(97,493, 1),
Point(401,483, 1),
Point(97,493, 1),
Point(401,483, 1),
Point(97,494, 1),
Point(400,483, 1),
Point(97,494, 1),
Point(400,483, 1),
Point(97,495, 1),
Point(400,483, 1),
Point(97,496, 1),
Point(401,483, 1),
Point(97,496, 1),
Point(401,483, 1),
Point(97,497, 1),
Point(402,483, 1),
Point(98,497, 1),
Point(402,483, 1),
Point(98,498, 1),
Point(403,483, 1),
Point(98,498, 1),
Point(403,483, 1),
Point(98,499, 1),
Point(403,483, 1),
Point(98,499, 1),
Point(404,483, 1),
Point(98,499, 1),
Point(404,483, 1),
Point(98,499, 1),
Point(404,483, 1),
Point(98,499, 1),
Point(404,483, 1),
Point(98,499, 1),
Point(403,484, 1),
Point(98,499, 1),
Point(403,485, 1),
Point(98,499, 1),
Point(402,485, 1),
Point(98,499, 1),
Point(402,485, 1),
Point(98,499, 1),
Point(402,485, 1),
Point(98,499, 1),
Point(402,485, 1),
Point(98,499, 1),
Point(402,486, 1),
Point(98,499, 1),
Point(402,486, 1),
Point(99,499, 1),
Point(402,486, 1),
Point(99,499, 1),
Point(402,486, 1),
Point(99,499, 1),
Point(402,486, 1),
Point(99,500, 1),
Point(402,486, 1),
Point(99,500, 1),
Point(402,486, 1),
Point(99,500, 1),
Point(402,486, 1),
Point(99,500, 1),
Point(402,486, 1),
Point(99,500, 1),
Point(402,486, 1),
Point(99,500, 1),
Point(402,486, 1),
Point(99,500, 1),
Point(402,486, 1),
Point(99,500, 1),
Point(402,486, 1),
Point(100,500, 1),
Point(402,486, 1),
Point(100,500, 1),
Point(402,486, 1),
Point(100,500, 1),
Point(402,486, 1),
Point(100,500, 1),
Point(402,486, 1),
Point(100,500, 1),
Point(402,486, 1),
Point(101,500, 1),
Point(402,486, 1),
Point(102,500, 1),
Point(402,486, 1),
Point(102,499, 1),
Point(402,486, 1),
Point(102,498, 1),
Point(402,486, 1),
Point(102,498, 1),
Point(401,486, 1),
Point(102,497, 1),
Point(399,487, 1),
Point(102,496, 1),
Point(397,487, 1),
Point(102,495, 1),
Point(396,487, 1),
Point(102,495, 1),
Point(395,487, 1),
Point(102,495, 1),
Point(393,487, 1),
Point(103,498, 1),
Point(393,488, 1),
])
SwipeRight = Template('SwipeRight', [
Point(119,496, 1),
Point(414,488, 1),
Point(119,495, 1),
Point(422,488, 1),
Point(118,492, 1),
Point(419,491, 1),
Point(117,486, 1),
Point(417,491, 1),
Point(117,479, 1),
Point(415,492, 1),
Point(116,477, 1),
Point(414,492, 1),
Point(116,477, 1),
Point(415,492, 1),
Point(115,475, 1),
Point(420,491, 1),
Point(114,473, 1),
Point(424,490, 1),
Point(114,473, 1),
Point(425,488, 1),
Point(116,474, 1),
Point(427,487, 1),
Point(117,474, 1),
Point(426,486, 1),
Point(118,474, 1),
Point(425,486, 1),
Point(118,475, 1),
Point(425,487, 1),
Point(118,476, 1),
Point(423,487, 1),
Point(119,477, 1),
Point(421,488, 1),
Point(119,478, 1),
Point(420,489, 1),
Point(119,479, 1),
Point(419,490, 1),
Point(119,480, 1),
Point(418,491, 1),
Point(119,480, 1),
Point(417,491, 1),
Point(119,480, 1),
Point(417,492, 1),
Point(118,480, 1),
Point(417,492, 1),
Point(118,480, 1),
Point(416,492, 1),
Point(118,480, 1),
Point(415,493, 1),
Point(118,480, 1),
Point(415,493, 1),
Point(118,480, 1),
Point(415,493, 1),
Point(118,480, 1),
Point(415,493, 1),
Point(117,480, 1),
Point(414,493, 1),
Point(117,480, 1),
Point(414,493, 1),
Point(117,480, 1),
Point(413,493, 1),
Point(117,480, 1),
Point(413,494, 1),
Point(116,480, 1),
Point(413,494, 1),
Point(116,479, 1),
Point(413,494, 1),
Point(116,479, 1),
Point(412,493, 1),
Point(116,479, 1),
Point(412,493, 1),
Point(116,479, 1),
Point(412,493, 1),
Point(116,479, 1),
Point(412,494, 1),
Point(116,479, 1),
Point(412,494, 1),
Point(116,479, 1),
Point(412,494, 1),
Point(116,479, 1),
Point(412,494, 1),
Point(116,479, 1),
Point(412,494, 1),
Point(116,479, 1),
Point(412,494, 1),
Point(116,480, 1),
Point(412,494, 1),
Point(116,480, 1),
Point(411,494, 1),
Point(116,480, 1),
Point(412,494, 1),
Point(116,480, 1),
Point(412,494, 1),
Point(116,480, 1),
Point(412,494, 1),
Point(116,480, 1),
Point(413,494, 1),
Point(115,480, 1),
Point(413,494, 1),
Point(115,480, 1),
Point(413,494, 1),
Point(115,480, 1),
Point(413,494, 1),
Point(115,481, 1),
Point(413,495, 1),
Point(116,481, 1),
Point(413,495, 1),
Point(116,481, 1),
Point(413,495, 1),
Point(116,481, 1),
Point(413,495, 1),
Point(117,482, 1),
Point(413,495, 1),
Point(118,482, 1),
Point(413,495, 1),
Point(119,483, 1),
Point(414,495, 1),
Point(203,362, 1),
Point(345,384, 1),
Point(212,352, 1),
Point(338,377, 1),
Point(214,351, 1),
Point(339,380, 1),
Point(207,332, 1),
Point(346,408, 1),
Point(206,330, 1),
Point(353,417, 1),
Point(206,329, 1),
Point(357,421, 1),
Point(214,335, 1),
Point(315,349, 1),
Point(216,337, 1),
Point(309,341, 1),
Point(216,337, 1),
Point(308,340, 1),
Point(229,318, 1),
Point(310,358, 1),
Point(231,315, 1),
Point(311,363, 1),
Point(232,314, 1),
Point(310,363, 1),
Point(232,314, 1),
Point(309,363, 1),
Point(233,294, 1),
Point(240,320, 1),
Point(233,290, 1),
Point(234,315, 1),
Point(233,289, 1),
Point(242,318, 1),
Point(229,284, 1),
Point(367,500, 1),
Point(228,281, 1),
Point(381,502, 1),
Point(227,280, 1),
Point(389,527, 1),
Point(227,281, 1),
Point(383,518, 1),
Point(226,279, 1),
Point(382,522, 1),
Point(226,277, 1),
Point(388,529, 1),
Point(226,274, 1),
Point(386,541, 1),
Point(225,273, 1),
Point(386,538, 1),
Point(225,273, 1),
Point(386,537, 1),
Point(225,273, 1),
Point(382,533, 1),
Point(225,273, 1),
Point(380,533, 1),
Point(225,273, 1),
Point(378,531, 1),
Point(225,273, 1),
Point(378,532, 1),
Point(225,272, 1),
Point(369,515, 1),
Point(224,273, 1),
Point(374,519, 1),
Point(223,273, 1),
Point(375,519, 1),
Point(223,273, 1),
Point(383,531, 1),
Point(222,273, 1),
Point(372,507, 1),
Point(222,273, 1),
Point(375,519, 1),
Point(221,273, 1),
Point(377,524, 1),
Point(220,274, 1),
Point(354,490, 1),
Point(220,274, 1),
Point(344,479, 1),
Point(219,274, 1),
Point(347,483, 1),
Point(219,274, 1),
Point(344,480, 1),
Point(219,274, 1),
Point(336,470, 1),
Point(219,275, 1),
Point(324,460, 1),
Point(219,275, 1),
Point(315,453, 1),
Point(217,275, 1),
Point(338,478, 1),
Point(216,275, 1),
Point(344,483, 1),
Point(215,275, 1),
Point(344,483, 1),
Point(213,274, 1),
Point(329,472, 1),
Point(211,273, 1),
Point(328,471, 1),
Point(210,273, 1),
Point(329,473, 1),
Point(205,273, 1),
Point(329,473, 1),
Point(203,273, 1),
Point(322,467, 1),
Point(195,273, 1),
Point(338,477, 1),
Point(192,273, 1),
Point(348,486, 1),
Point(191,273, 1),
Point(351,490, 1),
Point(187,273, 1),
Point(359,501, 1),
Point(185,272, 1),
Point(360,502, 1),
Point(184,272, 1),
Point(363,504, 1),
Point(184,272, 1),
Point(312,435, 1),
Point(184,271, 1),
Point(321,452, 1),
Point(184,271, 1),
Point(323,459, 1),
Point(171,271, 1),
Point(393,525, 1),
Point(167,271, 1),
Point(404,535, 1),
Point(166,271, 1),
Point(406,537, 1),
Point(146,273, 1),
Point(411,530, 1),
Point(144,274, 1),
Point(406,530, 1),
Point(143,274, 1),
Point(407,530, 1),
Point(143,275, 1),
Point(407,530, 1),
Point(136,275, 1),
Point(407,528, 1),
Point(136,274, 1),
Point(406,532, 1),
Point(136,274, 1),
Point(406,533, 1),
Point(125,274, 1),
Point(406,532, 1),
Point(122,274, 1),
Point(406,532, 1),
Point(121,274, 1),
Point(406,531, 1),
Point(112,274, 1),
Point(405,531, 1),
Point(109,274, 1),
Point(406,531, 1),
Point(105,269, 1),
Point(410,529, 1),
Point(101,267, 1),
Point(414,526, 1),
Point(102,262, 1),
Point(414,523, 1),
Point(97,262, 1),
Point(414,518, 1),
Point(94,262, 1),
Point(414,515, 1),
Point(93,262, 1),
Point(414,514, 1),
Point(94,262, 1),
Point(416,514, 1),
Point(94,261, 1),
Point(417,513, 1),
Point(94,260, 1),
Point(418,513, 1),
Point(93,262, 1),
Point(417,513, 1),
Point(93,263, 1),
Point(418,513, 1),
Point(93,264, 1),
Point(417,513, 1),
Point(91,263, 1),
Point(415,513, 1),
Point(90,262, 1),
Point(415,513, 1),
Point(90,261, 1),
Point(414,513, 1),
Point(89,263, 1),
Point(413,513, 1),
Point(88,265, 1),
Point(412,513, 1),
Point(88,266, 1),
Point(412,514, 1),
Point(87,266, 1),
Point(411,514, 1),
Point(87,273, 1),
Point(411,514, 1),
Point(87,276, 1),
Point(411,514, 1),
Point(86,277, 1),
Point(411,513, 1),
Point(86,281, 1),
Point(410,513, 1),
Point(86,282, 1),
Point(410,513, 1),
Point(86,283, 1),
Point(410,513, 1),
Point(86,284, 1),
Point(410,512, 1),
Point(85,284, 1),
Point(410,512, 1),
Point(85,285, 1),
Point(410,512, 1),
Point(84,286, 1),
Point(410,512, 1),
Point(84,287, 1),
Point(410,513, 1),
Point(83,290, 1),
Point(410,513, 1),
Point(83,292, 1),
Point(410,513, 1),
Point(83,293, 1),
Point(410,513, 1),
Point(83,295, 1),
Point(410,513, 1),
Point(83,296, 1),
Point(410,513, 1),
Point(83,296, 1),
Point(410,513, 1),
Point(83,299, 1),
Point(410,513, 1),
Point(83,301, 1),
Point(410,513, 1),
Point(83,301, 1),
Point(410,513, 1),
Point(82,310, 1),
Point(410,513, 1),
Point(82,313, 1),
Point(410,513, 1),
Point(82,314, 1),
Point(410,514, 1),
Point(81,327, 1),
Point(410,514, 1),
Point(80,331, 1),
Point(411,513, 1),
])
ZoomIn = Template('ZoomIn', [
Point(94,479, 1),
Point(401,502, 1),
Point(101,486, 1),
Point(396,497, 1),
Point(103,493, 1),
Point(391,495, 1),
Point(102,493, 1),
Point(390,493, 1),
Point(103,490, 1),
Point(389,488, 1),
Point(104,488, 1),
Point(390,487, 1),
Point(103,487, 1),
Point(390,487, 1),
Point(103,486, 1),
Point(391,488, 1),
Point(102,486, 1),
Point(390,488, 1),
Point(102,485, 1),
Point(390,489, 1),
Point(102,485, 1),
Point(390,489, 1),
Point(102,484, 1),
Point(390,489, 1),
Point(102,484, 1),
Point(390,489, 1),
Point(102,484, 1),
Point(390,489, 1),
Point(102,484, 1),
Point(390,489, 1),
Point(102,484, 1),
Point(389,489, 1),
Point(102,484, 1),
Point(389,489, 1),
Point(102,483, 1),
Point(389,489, 1),
Point(103,483, 1),
Point(388,489, 1),
Point(103,483, 1),
Point(388,489, 1),
Point(103,483, 1),
Point(388,489, 1),
Point(103,483, 1),
Point(388,489, 1),
Point(103,483, 1),
Point(388,489, 1),
Point(103,483, 1),
Point(388,489, 1),
Point(103,484, 1),
Point(388,489, 1),
Point(103,484, 1),
Point(388,489, 1),
Point(103,484, 1),
Point(389,489, 1),
Point(103,485, 1),
Point(389,489, 1),
Point(103,485, 1),
Point(389,489, 1),
Point(103,485, 1),
Point(389,489, 1),
Point(103,486, 1),
Point(389,489, 1),
Point(104,487, 1),
Point(389,489, 1),
Point(104,487, 1),
Point(389,489, 1),
Point(104,488, 1),
Point(389,489, 1),
Point(104,488, 1),
Point(389,488, 1),
Point(104,489, 1),
Point(390,488, 1),
Point(104,490, 1),
Point(390,488, 1),
Point(104,491, 1),
Point(390,488, 1),
Point(104,491, 1),
Point(390,488, 1),
Point(104,491, 1),
Point(390,488, 1),
Point(104,491, 1),
Point(390,488, 1),
Point(104,492, 1),
Point(390,488, 1),
Point(104,492, 1),
Point(390,488, 1),
Point(104,492, 1),
Point(390,488, 1),
Point(104,492, 1),
Point(390,488, 1),
Point(104,492, 1),
Point(390,488, 1),
Point(105,492, 1),
Point(390,488, 1),
Point(105,492, 1),
Point(390,487, 1),
Point(105,491, 1),
Point(389,487, 1),
Point(105,491, 1),
Point(389,487, 1),
Point(105,491, 1),
Point(390,487, 1),
Point(105,491, 1),
Point(390,487, 1),
Point(105,491, 1),
Point(390,487, 1),
Point(105,491, 1),
Point(390,486, 1),
Point(105,492, 1),
Point(391,486, 1),
Point(105,492, 1),
Point(392,486, 1),
Point(105,492, 1),
Point(393,486, 1),
Point(105,493, 1),
Point(393,486, 1),
Point(186,364, 1),
Point(276,356, 1),
Point(193,352, 1),
Point(264,342, 1),
Point(193,348, 1),
Point(263,340, 1),
Point(183,334, 1),
Point(297,343, 1),
Point(181,333, 1),
Point(309,344, 1),
Point(174,348, 1),
Point(326,379, 1),
Point(171,349, 1),
Point(331,389, 1),
Point(171,348, 1),
Point(333,395, 1),
Point(195,299, 1),
Point(329,327, 1),
Point(199,293, 1),
Point(328,320, 1),
Point(200,293, 1),
Point(328,320, 1),
Point(210,280, 1),
Point(306,271, 1),
Point(214,283, 1),
Point(302,267, 1),
Point(215,282, 1),
Point(299,265, 1),
Point(212,270, 1),
Point(301,260, 1),
Point(210,267, 1),
Point(301,258, 1),
Point(209,266, 1),
Point(301,257, 1),
Point(208,268, 1),
Point(306,263, 1),
Point(205,269, 1),
Point(309,264, 1),
Point(202,269, 1),
Point(312,265, 1),
Point(199,271, 1),
Point(311,254, 1),
Point(196,273, 1),
Point(309,257, 1),
Point(195,274, 1),
Point(309,260, 1),
Point(192,275, 1),
Point(306,262, 1),
Point(190,275, 1),
Point(305,263, 1),
Point(189,276, 1),
Point(309,264, 1),
Point(188,276, 1),
Point(311,264, 1),
Point(188,276, 1),
Point(311,264, 1),
Point(188,276, 1),
Point(312,264, 1),
Point(187,277, 1),
Point(315,265, 1),
Point(186,277, 1),
Point(316,265, 1),
Point(186,277, 1),
Point(317,265, 1),
Point(184,277, 1),
Point(312,265, 1),
Point(184,277, 1),
Point(309,264, 1),
Point(184,277, 1),
Point(306,263, 1),
Point(182,277, 1),
Point(306,262, 1),
Point(179,278, 1),
Point(316,263, 1),
Point(178,278, 1),
Point(320,264, 1),
Point(178,277, 1),
Point(322,264, 1),
Point(178,278, 1),
Point(323,266, 1),
Point(178,278, 1),
Point(323,267, 1),
Point(178,276, 1),
Point(326,266, 1),
Point(177,275, 1),
Point(329,266, 1),
Point(177,274, 1),
Point(331,267, 1),
Point(170,279, 1),
Point(340,268, 1),
Point(169,279, 1),
Point(344,268, 1),
Point(169,279, 1),
Point(344,267, 1),
Point(161,282, 1),
Point(359,268, 1),
Point(158,285, 1),
Point(362,262, 1),
Point(139,285, 1),
Point(382,253, 1),
Point(133,285, 1),
Point(380,249, 1),
Point(130,283, 1),
Point(378,221, 1),
Point(123,283, 1),
Point(398,249, 1),
Point(120,284, 1),
Point(406,273, 1),
Point(119,285, 1),
Point(407,274, 1),
Point(120,285, 1),
Point(407,274, 1),
Point(114,285, 1),
Point(402,272, 1),
Point(112,284, 1),
Point(399,272, 1),
Point(110,283, 1),
Point(398,271, 1),
Point(101,273, 1),
Point(402,274, 1),
Point(98,269, 1),
Point(403,272, 1),
Point(97,267, 1),
Point(403,267, 1),
Point(92,258, 1),
Point(436,446, 1),
Point(90,257, 1),
Point(441,459, 1),
Point(89,258, 1),
Point(442,443, 1),
Point(83,258, 1),
Point(437,317, 1),
Point(82,259, 1),
Point(433,265, 1),
Point(81,259, 1),
Point(431,270, 1),
Point(81,259, 1),
Point(431,375, 1),
Point(79,251, 1),
Point(425,274, 1),
Point(78,250, 1),
Point(420,271, 1),
Point(78,249, 1),
Point(421,365, 1),
Point(78,249, 1),
Point(421,379, 1),
Point(78,248, 1),
Point(424,456, 1),
Point(77,247, 1),
Point(425,491, 1),
Point(76,247, 1),
Point(426,476, 1),
Point(76,246, 1),
Point(425,481, 1),
Point(76,247, 1),
Point(425,495, 1),
Point(77,249, 1),
Point(425,503, 1),
Point(77,249, 1),
Point(425,506, 1),
Point(77,249, 1),
Point(425,507, 1),
Point(77,248, 1),
Point(425,506, 1),
Point(77,248, 1),
Point(424,506, 1),
Point(82,265, 1),
Point(424,504, 1),
Point(85,272, 1),
Point(423,502, 1),
Point(86,274, 1),
Point(423,501, 1),
Point(87,271, 1),
Point(418,505, 1),
Point(87,270, 1),
Point(414,506, 1),
Point(88,269, 1),
Point(413,506, 1),
Point(88,253, 1),
Point(403,506, 1),
Point(88,249, 1),
Point(399,506, 1),
Point(88,248, 1),
Point(397,505, 1),
Point(103,312, 1),
Point(397,503, 1),
Point(109,320, 1),
Point(395,501, 1),
Point(111,320, 1),
Point(393,499, 1),
Point(101,334, 1),
Point(392,490, 1),
Point(96,337, 1),
Point(392,484, 1),
Point(94,452, 1),
Point(393,486, 1),
Point(94,456, 1),
Point(393,487, 1),
Point(92,441, 1),
Point(395,489, 1),
Point(97,488, 1),
Point(395,488, 1),
Point(100,492, 1),
Point(395,488, 1),
Point(103,494, 1),
Point(395,488, 1),
Point(107,492, 1),
Point(393,488, 1),
Point(108,493, 1),
Point(393,488, 1),
Point(108,492, 1),
Point(392,487, 1),
Point(108,490, 1),
Point(392,487, 1),
Point(108,490, 1),
Point(392,487, 1),
Point(108,490, 1),
Point(391,487, 1),
Point(107,490, 1),
Point(391,487, 1),
Point(107,491, 1),
Point(391,487, 1),
Point(107,491, 1),
Point(391,487, 1),
Point(107,491, 1),
Point(391,487, 1),
Point(107,492, 1),
Point(391,486, 1),
Point(107,492, 1),
Point(392,486, 1),
Point(107,492, 1),
Point(392,486, 1),
Point(107,492, 1),
Point(392,486, 1),
Point(107,491, 1),
Point(392,486, 1),
Point(107,491, 1),
Point(393,486, 1),
Point(107,493, 1),
Point(394,486, 1),
Point(107,493, 1),
Point(394,486, 1),
Point(107,493, 1),
Point(396,486, 1),
Point(107,493, 1),
Point(397,486, 1),
Point(107,493, 1),
Point(397,486, 1),
Point(106,493, 1),
Point(397,486, 1),
Point(105,493, 1),
Point(397,486, 1),
Point(106,493, 1),
Point(397,486, 1),
Point(106,494, 1),
Point(397,486, 1),
Point(106,494, 1),
Point(397,486, 1),
Point(106,494, 1),
Point(397,486, 1),
Point(106,494, 1),
Point(397,486, 1),
Point(106,493, 1),
Point(397,486, 1),
Point(106,493, 1),
Point(397,486, 1),
Point(106,493, 1),
Point(397,486, 1),
Point(106,492, 1),
Point(397,486, 1),
Point(105,492, 1),
Point(397,486, 1),
Point(105,492, 1),
Point(397,486, 1),
Point(105,491, 1),
Point(397,486, 1),
Point(105,490, 1),
Point(396,486, 1),
Point(105,490, 1),
Point(396,486, 1),
Point(105,489, 1),
Point(396,486, 1),
Point(105,489, 1),
Point(396,486, 1),
Point(105,489, 1),
Point(396,486, 1),
Point(105,489, 1),
Point(395,487, 1),
Point(105,489, 1),
Point(395,487, 1),
Point(105,488, 1),
Point(394,487, 1),
Point(105,487, 1),
Point(393,486, 1),
])
ZoomOut = Template('ZoomOut', [
Point(104,487, 1),
Point(393,487, 1),
Point(104,487, 1),
Point(393,487, 1),
Point(104,487, 1),
Point(393,487, 1),
Point(103,487, 1),
Point(393,487, 1),
Point(103,487, 1),
Point(393,487, 1),
Point(103,487, 1),
Point(392,487, 1),
Point(103,487, 1),
Point(392,487, 1),
Point(103,487, 1),
Point(392,487, 1),
Point(103,487, 1),
Point(392,487, 1),
Point(102,487, 1),
Point(392,487, 1),
Point(102,487, 1),
Point(392,486, 1),
Point(102,487, 1),
Point(392,486, 1),
Point(102,487, 1),
Point(392,486, 1),
Point(102,487, 1),
Point(392,486, 1),
Point(102,487, 1),
Point(392,486, 1),
Point(102,487, 1),
Point(392,486, 1),
Point(102,487, 1),
Point(392,486, 1),
Point(102,486, 1),
Point(393,486, 1),
Point(102,486, 1),
Point(393,486, 1),
Point(102,486, 1),
Point(393,486, 1),
Point(102,486, 1),
Point(393,486, 1),
Point(102,485, 1),
Point(393,486, 1),
Point(102,485, 1),
Point(393,486, 1),
Point(102,485, 1),
Point(393,486, 1),
Point(102,485, 1),
Point(394,485, 1),
Point(102,485, 1),
Point(394,485, 1),
Point(102,486, 1),
Point(395,485, 1),
Point(102,486, 1),
Point(395,485, 1),
Point(102,486, 1),
Point(395,485, 1),
Point(102,486, 1),
Point(395,485, 1),
Point(102,486, 1),
Point(395,485, 1),
Point(102,486, 1),
Point(395,485, 1),
Point(146,356, 1),
Point(378,412, 1),
Point(153,348, 1),
Point(380,409, 1),
Point(155,337, 1),
Point(373,337, 1),
Point(157,336, 1),
Point(372,331, 1),
Point(156,335, 1),
Point(372,333, 1),
Point(157,324, 1),
Point(370,326, 1),
Point(157,322, 1),
Point(369,327, 1),
Point(158,322, 1),
Point(369,329, 1),
Point(157,320, 1),
Point(369,310, 1),
Point(156,319, 1),
Point(370,307, 1),
Point(155,319, 1),
Point(371,306, 1),
Point(155,302, 1),
Point(375,295, 1),
Point(156,298, 1),
Point(378,287, 1),
Point(156,298, 1),
Point(378,290, 1),
Point(147,289, 1),
Point(379,278, 1),
Point(147,284, 1),
Point(378,267, 1),
Point(147,287, 1),
Point(378,259, 1),
Point(147,283, 1),
Point(378,266, 1),
Point(136,279, 1),
Point(381,260, 1),
Point(135,278, 1),
Point(382,259, 1),
Point(134,279, 1),
Point(381,258, 1),
Point(130,279, 1),
Point(384,258, 1),
Point(127,279, 1),
Point(385,258, 1),
Point(126,280, 1),
Point(385,258, 1),
Point(125,276, 1),
Point(386,256, 1),
Point(124,274, 1),
Point(390,253, 1),
Point(123,268, 1),
Point(390,251, 1),
Point(123,266, 1),
Point(390,251, 1),
Point(122,266, 1),
Point(390,250, 1),
Point(122,265, 1),
Point(391,244, 1),
Point(121,265, 1),
Point(392,243, 1),
Point(121,265, 1),
Point(392,242, 1),
Point(121,265, 1),
Point(393,242, 1),
Point(121,265, 1),
Point(393,242, 1),
Point(121,265, 1),
Point(393,242, 1),
Point(121,265, 1),
Point(394,242, 1),
Point(121,266, 1),
Point(395,243, 1),
Point(121,266, 1),
Point(395,242, 1),
Point(121,267, 1),
Point(395,241, 1),
Point(121,267, 1),
Point(395,241, 1),
Point(121,268, 1),
Point(395,241, 1),
Point(121,268, 1),
Point(396,242, 1),
Point(121,268, 1),
Point(396,243, 1),
Point(121,269, 1),
Point(396,243, 1),
Point(121,269, 1),
Point(396,243, 1),
Point(120,269, 1),
Point(397,245, 1),
Point(120,269, 1),
Point(397,246, 1),
Point(120,269, 1),
Point(397,246, 1),
Point(125,269, 1),
Point(397,246, 1),
Point(130,268, 1),
Point(398,251, 1),
Point(137,269, 1),
Point(405,265, 1),
Point(137,271, 1),
Point(406,266, 1),
Point(138,272, 1),
Point(405,264, 1),
Point(154,271, 1),
Point(416,463, 1),
Point(157,271, 1),
Point(420,444, 1),
Point(157,271, 1),
Point(418,400, 1),
Point(163,272, 1),
Point(404,262, 1),
Point(164,274, 1),
Point(395,246, 1),
Point(165,274, 1),
Point(391,250, 1),
Point(181,275, 1),
Point(367,156, 1),
Point(186,275, 1),
Point(361,137, 1),
Point(185,277, 1),
Point(358,149, 1),
Point(205,280, 1),
Point(356,206, 1),
Point(213,280, 1),
Point(355,225, 1),
Point(214,280, 1),
Point(346,226, 1),
Point(214,291, 1),
Point(350,241, 1),
Point(215,292, 1),
Point(352,249, 1),
Point(216,294, 1),
Point(352,257, 1),
Point(221,295, 1),
Point(354,258, 1),
Point(225,294, 1),
Point(353,256, 1),
Point(225,294, 1),
Point(353,256, 1),
Point(225,295, 1),
Point(351,272, 1),
Point(234,293, 1),
Point(350,268, 1),
Point(238,286, 1),
Point(339,261, 1),
Point(237,287, 1),
Point(340,261, 1),
Point(242,283, 1),
Point(335,235, 1),
Point(241,283, 1),
Point(334,232, 1),
Point(240,285, 1),
Point(336,235, 1),
Point(240,287, 1),
Point(338,236, 1),
Point(241,287, 1),
Point(338,235, 1),
Point(241,288, 1),
Point(341,229, 1),
Point(240,290, 1),
Point(346,230, 1),
Point(240,292, 1),
Point(348,228, 1),
Point(228,316, 1),
Point(363,255, 1),
Point(224,321, 1),
Point(367,262, 1),
Point(222,322, 1),
Point(368,263, 1),
Point(222,321, 1),
Point(366,260, 1),
Point(222,321, 1),
Point(365,259, 1),
Point(222,322, 1),
Point(365,258, 1),
Point(227,294, 1),
Point(324,241, 1),
Point(234,289, 1),
Point(316,236, 1),
Point(234,291, 1),
Point(313,231, 1),
Point(241,286, 1),
Point(311,234, 1),
Point(243,288, 1),
Point(311,236, 1),
Point(244,289, 1),
Point(311,238, 1),
Point(259,272, 1),
Point(316,261, 1),
Point(263,267, 1),
Point(321,267, 1),
Point(264,265, 1),
Point(323,269, 1),
Point(245,315, 1),
Point(323,314, 1),
Point(239,324, 1),
Point(325,321, 1),
Point(234,326, 1),
Point(324,318, 1),
Point(223,346, 1),
Point(339,333, 1),
Point(219,351, 1),
Point(347,338, 1),
Point(216,354, 1),
Point(350,340, 1),
Point(218,362, 1),
Point(352,386, 1),
Point(214,366, 1),
Point(352,392, 1),
Point(212,368, 1),
Point(353,395, 1),
Point(121,471, 1),
Point(381,472, 1),
Point(112,479, 1),
Point(387,478, 1),
Point(111,480, 1),
Point(388,479, 1),
Point(111,480, 1),
Point(388,480, 1),
Point(112,481, 1),
Point(389,480, 1),
Point(110,479, 1),
Point(388,479, 1),
Point(108,477, 1),
Point(387,479, 1),
Point(107,476, 1),
Point(387,479, 1),
Point(107,476, 1),
Point(387,479, 1),
Point(108,475, 1),
Point(386,481, 1),
Point(108,474, 1),
Point(385,482, 1),
Point(108,473, 1),
Point(385,482, 1),
Point(108,473, 1),
Point(385,483, 1),
Point(107,473, 1),
Point(385,483, 1),
Point(107,473, 1),
Point(385,484, 1),
Point(107,473, 1),
Point(385,485, 1),
Point(106,473, 1),
Point(384,485, 1),
Point(106,472, 1),
Point(384,484, 1),
Point(105,472, 1),
Point(384,484, 1),
Point(105,472, 1),
Point(384,484, 1),
Point(105,472, 1),
Point(384,484, 1),
Point(104,472, 1),
Point(384,484, 1),
Point(104,472, 1),
Point(384,484, 1),
Point(104,472, 1),
Point(384,483, 1),
Point(104,471, 1),
Point(384,483, 1),
Point(104,471, 1),
Point(384,483, 1),
Point(103,471, 1),
Point(383,483, 1),
Point(103,471, 1),
Point(383,483, 1),
Point(103,471, 1),
Point(383,483, 1),
Point(103,471, 1),
Point(383,483, 1),
Point(102,472, 1),
Point(383,484, 1),
Point(102,472, 1),
Point(383,484, 1),
Point(102,472, 1),
Point(383,484, 1),
Point(102,473, 1),
Point(383,484, 1),
Point(102,473, 1),
Point(383,483, 1),
Point(102,473, 1),
Point(383,483, 1),
Point(102,473, 1),
Point(383,483, 1),
Point(102,473, 1),
Point(383,483, 1),
Point(102,473, 1),
Point(383,483, 1),
Point(102,474, 1),
Point(383,483, 1),
Point(102,474, 1),
Point(383,483, 1),
Point(102,474, 1),
Point(384,483, 1),
Point(102,474, 1),
Point(384,483, 1),
Point(102,474, 1),
Point(385,483, 1),
Point(103,474, 1),
Point(386,482, 1),
Point(103,474, 1),
Point(386,482, 1),
Point(103,474, 1),
Point(387,482, 1),
Point(103,474, 1),
Point(387,482, 1),
Point(103,474, 1),
Point(387,482, 1),
Point(103,474, 1),
Point(387,482, 1),
Point(103,474, 1),
Point(387,482, 1),
Point(103,473, 1),
Point(388,482, 1),
Point(103,473, 1),
Point(388,482, 1),
])
recognizer = Recognizer([SwipeLeft,SwipeRight,ZoomIn,ZoomOut])
all_macs=[]
class HCIServer:
    """Main server class — one thread per connected client."""

    def __init__(self):
        self.face_cascade = cv2.CascadeClassifier(
            cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
        )
        self._sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self._sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        self._sock.bind((SERVER_HOST, SERVER_PORT))
        self._sock.listen(5)
        print(f"[SERVER] Listening on {SERVER_HOST}:{SERVER_PORT}")
        print(f"[SERVER] {TUIO_NOTE}\n")

    # Step 2 · Bluetooth Scan

    async def _ble_scan(self) -> str | None:
        print("[BT] Scanning for Bluetooth devices (5 s)…")
        devices = await BleakScanner.discover(timeout=5.0)

        if len(devices) > 0:
            first = devices[0]
            print(f"[BT] Using first detected device: {first.name} [{first.address}]")
            return first.address

        print("[BT] No devices found")
        return None

    def scan_bluetooth(self) -> str | None:
        import threading
        result_holder = [None]

        def run_scan():
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            try:
                result_holder[0] = loop.run_until_complete(self._ble_scan())
            except Exception as e:
                print(f"[BT] Error: {e}")
            finally:
                loop.close()

        t = threading.Thread(target=run_scan, daemon=True)
        t.start()
        t.join(timeout=15)
        return result_holder[0]

server = HCIServer()
address = None
address = server.scan_bluetooth()
cap = cv2.VideoCapture(0)
user_login = 0
flag_bluetooth = 0
while cap.isOpened():
    _, frame = cap.read()
    msg=''
    if user_login == 0 and address is not None:
        print("Sending MAC:", address)
        
        conn.send(address.encode('utf-8'))
        user_login = 1
    try:
        
     
        f_frame = cv2.resize(frame, (480, 320))
        frame_rgb = cv2.cvtColor(f_frame, cv2.COLOR_BGR2RGB)
        frame_count += 1
        # if frame_count % 60 == 0:
        #     face_encodings = DeepFace.represent(
        #         f_frame,
        #         model_name="Facenet",
        #         enforce_detection=False
        #     )
        #     if len(face_encodings) > 0:
        #         face_id = tuple(face_encodings[0]["embedding"])
        #         match = None
        #         for k in face_ids:
        #             if np.linalg.norm(np.array(face_id) - np.array(k)) < 15:
        #                 match = face_ids[k]
        #                 msg = "Known face recognized: " + match
        #                 break
        #         if match is None:
        #             face_ids[face_id] = "Person " + str(len(face_ids) + 1)
        #             msg = "New face detected: " + face_ids[face_id]
        #             print("New face detected: " + face_ids[face_id])
        #         else:
        #             print("Known face recognized: " + match)
        results = holistic.process(frame_rgb)
        annotated_image = f_frame.copy()
        image_height, image_width, _ = frame_rgb.shape
        image_hight, image_width, _ = frame.shape

        x=(int(results.pose_landmarks.landmark[mp_pose.PoseLandmark.RIGHT_WRIST].x * image_width))
        y=(int(results.pose_landmarks.landmark[mp_pose.PoseLandmark.RIGHT_WRIST].y * image_hight))
        Allpoints.append(Point(x,y,1))

        x=(int(results.pose_landmarks.landmark[mp_pose.PoseLandmark.LEFT_WRIST].x * image_width))
        y=(int(results.pose_landmarks.landmark[mp_pose.PoseLandmark.LEFT_WRIST].y * image_hight))
        Allpoints.append(Point(x,y,1))

        if frame_count%30==0:
            frame_count=0
            result = recognizer.recognize(Allpoints)
            if result[0] != None  :
                print(result)
                msg+=result[0]
                
            
            Allpoints.clear()
        # x=int(results.pose.landmark[mp_pose.PoseLandmark.RIGHT_WRIST].x * image_width)
        # y=int(results.pose.landmark[mp_pose.PoseLandmark.RIGHT_WRIST].y * image_height)
        for face_id_key, name in face_ids.items():
            cv2.putText(annotated_image, name, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        mp_drawing.draw_landmarks(annotated_image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
        mp_drawing.draw_landmarks(annotated_image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
        mp_drawing.draw_landmarks(
            annotated_image,
            results.face_landmarks,
            mp_holistic.FACEMESH_TESSELATION,
            landmark_drawing_spec=None,
            connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_tesselation_style()
        )
        mp_drawing.draw_landmarks(
            annotated_image,
            results.pose_landmarks,
            mp_holistic.POSE_CONNECTIONS,
            landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style()
        )
        cv2.imshow("Output", annotated_image)
        #logic to send msg to unity
        if msg!='' and msg != old_msg:  # only send when there's actually something
            conn.send(msg.encode('utf-8'))
            
        old_msg = msg

        if msg == pickle.dumps("exit"):
            break
    except Exception as e:
        print(e)
    if cv2.waitKey(1) == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

Device Connected
[SERVER] Listening on 0.0.0.0:5000
[SERVER] Remember to start your TUIO simulator/tracker on port 3333

[BT] Scanning for Bluetooth devices (5 s)…
[BT] Using first detected device: None [FA:21:21:96:33:D1]
Sending MAC: FA:21:21:96:33:D1
('SwipeRight', 0.1156647513843464)
('SwipeRight', 0.0025333384539979464)
('SwipeRight', 0.045933729205341334)
('SwipeRight', 0.015430703847645)
('SwipeRight', 0.06714864572755341)
('SwipeRight', 0.17421259989476)
('SwipeRight', 0.006554748711915637)


KeyboardInterrupt: 

In [1]:
import mediapipe as mp
print(mp.__version__)


0.10.9
